Creates new introductory graphics that feature both with and without wind and a day night variable. Saved to the intro_graphics folder. Graphics included are: heatmaps (both raw and difference from mean), least squares output, wind projections, and weather distributions

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.colors import Normalize, TwoSlopeNorm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

# ============================================================
# PARAMETERS
# ============================================================
TEAM = 'SF'
RUN_ALL_TEAMS = True
SEASON_START = None
SEASON_END = None

DATASETS_DIR = os.path.join('..', 'data')
OUTPUT_ROOT = 'cleaned_analysis'
LEAGUE_FILE = os.path.join(DATASETS_DIR, 'league_weather_2021_2025.csv')

params_df = pd.read_csv('team_parameters.csv')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

if os.path.exists(LEAGUE_FILE):
    league_data = pd.read_csv(LEAGUE_FILE)
    league_data['game_date'] = pd.to_datetime(league_data['game_date'])
    print(f'League data loaded: {len(league_data)} games')
else:
    raise FileNotFoundError(f'League weather file not found: {LEAGUE_FILE}')

teams_to_process = params_df['team_code'].tolist() if RUN_ALL_TEAMS else [TEAM]
print(f'Teams to process: {teams_to_process}')
print(f'Output root: {os.path.abspath(OUTPUT_ROOT)}')

In [ ]:
# ============================================================
# MODEL / PLOT CONFIGURATION
# ============================================================
DEPENDENT_VARS = {
    'away_runs_scored': 'Away Runs Scored',
    'away_bat_k': 'Away Strikeouts',
    'away_bat_hr_h_ratio': 'Away HR:H Ratio',
}

MODEL_SPECS = {
    'with_wind': {
        'label': 'With Wind + Day/Night Feature',
        'ivars': ['temp_f', 'rhum', 'pres', 'prcp', 'is_night', 'wspd_mph', 'wind_cf', 'wind_lcf', 'wind_rcf'],
        'center_vars': ['temp_f', 'rhum', 'pres', 'prcp', 'wspd_mph', 'wind_cf', 'wind_lcf', 'wind_rcf'],
    },
    'without_wind': {
        'label': 'Without Wind + Day/Night Feature',
        'ivars': ['temp_f', 'rhum', 'pres', 'prcp', 'is_night'],
        'center_vars': ['temp_f', 'rhum', 'pres', 'prcp'],
    },
}

IV_DISPLAY_NAMES = {
    'temp_f': 'Temp (F)',
    'rhum': 'Humidity (%)',
    'pres': 'Pressure (hPa)',
    'prcp': 'Precip (mm)',
    'wspd_mph': 'Wind Speed (mph)',
    'wind_cf': 'Wind to CF',
    'wind_lcf': 'Wind to LCF',
    'wind_rcf': 'Wind to RCF',
    'is_night': 'Night Game (1=yes)',
    'const': 'Intercept',
}

WEATHER_VARS = {
    'temp_f': {'label': 'Temperature', 'unit': '°F', 'n_bins': 5, 'fmt': '.0f'},
    'wspd_mph': {'label': 'Wind Speed', 'unit': ' mph', 'n_bins': 5, 'fmt': '.0f'},
    'rhum': {'label': 'Humidity', 'unit': '%', 'n_bins': 5, 'fmt': '.0f'},
    'pres': {'label': 'Pressure', 'unit': ' hPa', 'n_bins': 5, 'fmt': '.0f'},
}

BASEBALL_STATS = {
    'away_runs_scored': {'label': 'Away Runs', 'fmt': '.1f'},
    'away_bat_hr': {'label': 'Away Home Runs', 'fmt': '.2f'},
    'away_bat_k': {'label': 'Away Strikeouts', 'fmt': '.1f'},
    'away_bat_bb': {'label': 'Away Walks', 'fmt': '.1f'},
    'away_bat_hr_h_ratio': {'label': 'Away HR:H Ratio', 'fmt': '.3f'},
}

WIND_PROJ_VARS = {
    'wind_cf': {'label': 'Wind to CF', 'color': '#c0392b'},
    'wind_lcf': {'label': 'Wind to LCF', 'color': '#2980b9'},
    'wind_rcf': {'label': 'Wind to RCF', 'color': '#27ae60'},
}

MIN_GAMES_PER_BIN = 5
N_WIND_BINS = 10

print('Configuration loaded.')

In [ ]:
def day_night_dataset_file(dataset_file):
    base, ext = os.path.splitext(dataset_file)
    return f'{base}_day_night{ext}'


def prepare_team_data(team_row):
    dataset_file = day_night_dataset_file(team_row['dataset_file'])
    dataset_path = os.path.join(DATASETS_DIR, dataset_file)
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f'Dataset not found: {dataset_path}')

    df = pd.read_csv(dataset_path)
    df['game_date'] = pd.to_datetime(df['game_date'])

    if 'day_night' not in df.columns:
        if 'start_hour' not in df.columns:
            raise ValueError(f'{dataset_file} is missing both day_night and start_hour')
        df['day_night'] = np.where(pd.to_numeric(df['start_hour'], errors='coerce') < 17, 'day', 'night')

    df['day_night'] = df['day_night'].astype(str).str.strip().str.lower()
    df.loc[~df['day_night'].isin(['day', 'night']), 'day_night'] = pd.NA
    df['is_night'] = np.where(df['day_night'] == 'night', 1.0, 0.0)
    df.loc[df['day_night'].isna(), 'is_night'] = np.nan

    season_start = int(team_row['data_start_year']) if SEASON_START is None else int(SEASON_START)
    season_end = int(team_row['data_end_year']) if SEASON_END is None else int(SEASON_END)

    df = df[(df['season'] >= season_start) & (df['season'] <= season_end)].copy()
    return df, dataset_file, season_start, season_end


def plot_weather_distribution(team_window, league_window, col, xlabel, unit, stadium_name, graphics_dir, show=True):
    team_vals = team_window[col].dropna()
    league_vals = league_window[col].dropna()
    if len(team_vals) == 0 or len(league_vals) == 0:
        return None

    if col == 'rhum':
        bins = np.linspace(0, 100, 35)
    elif col == 'wspd_mph':
        bins = np.linspace(0, max(team_vals.max(), league_vals.max()) + 1, 35)
    else:
        bins = np.linspace(min(team_vals.min(), league_vals.min()) - 2, max(team_vals.max(), league_vals.max()) + 2, 40 if col == 'temp_f' else 35)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(league_vals, bins=bins, density=True, alpha=0.5, color='steelblue', edgecolor='white', linewidth=0.5, label=f'League Avg (mean: {league_vals.mean():.1f}{unit})')
    ax.hist(team_vals, bins=bins, density=True, alpha=0.6, color='firebrick', edgecolor='white', linewidth=0.5, label=f'{stadium_name} (mean: {team_vals.mean():.1f}{unit})')
    ax.axvline(league_vals.mean(), color='steelblue', linestyle='--', linewidth=1.5)
    ax.axvline(team_vals.mean(), color='firebrick', linestyle='--', linewidth=1.5)
    ax.set_xlabel(f'{xlabel} ({unit.strip()})', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'{xlabel} Distribution: {stadium_name} vs League Average (2021-2025)', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    filepath = os.path.join(graphics_dir, f'{col}_distribution.png')
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filepath


def run_ols(df, dep_var, indep_vars, center_vars):
    cols_needed = [dep_var] + indep_vars
    reg_df = df[cols_needed].dropna().copy()
    n_dropped = len(df) - len(reg_df)
    if len(reg_df) == 0:
        raise ValueError('No rows remain after dropping NA values for regression')

    iv_means = {}
    for col in center_vars:
        iv_means[col] = reg_df[col].mean()
        reg_df[col] = reg_df[col] - iv_means[col]

    X = sm.add_constant(reg_df[indep_vars])
    y = reg_df[dep_var]
    model = sm.OLS(y, X).fit()
    return model, n_dropped, iv_means


def results_to_dataframe(model):
    summary_df = pd.DataFrame({
        'Variable': [IV_DISPLAY_NAMES.get(v, v) for v in model.params.index],
        'Coefficient': model.params.values,
        'Std Error': model.bse.values,
        't-stat': model.tvalues.values,
        'P-value': model.pvalues.values,
    })

    def sig_stars(p):
        if p < 0.001:
            return '***'
        if p < 0.01:
            return '**'
        if p < 0.05:
            return '*'
        if p < 0.10:
            return '.'
        return ''

    summary_df['Sig'] = summary_df['P-value'].apply(sig_stars)
    summary_df['Coefficient'] = summary_df['Coefficient'].round(4)
    summary_df['Std Error'] = summary_df['Std Error'].round(4)
    summary_df['t-stat'] = summary_df['t-stat'].round(3)
    summary_df['P-value'] = summary_df['P-value'].round(4)
    return summary_df


def render_regression_table(results_df, model, dep_label, model_label, n_obs, n_dropped, stadium_name, season_start, season_end, filepath, show=True):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axis('off')
    title = f'OLS Regression: {dep_label}\n{stadium_name} — {model_label} ({season_start}-{season_end})'
    ax.set_title(title, fontsize=13, fontweight='bold', pad=20, loc='left')

    table = ax.table(cellText=results_df.values.tolist(), colLabels=results_df.columns.tolist(), cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.45)

    for j in range(len(results_df.columns)):
        table[0, j].set_facecolor('#2c3e50')
        table[0, j].set_text_props(color='white', fontweight='bold')

    for i in range(1, len(results_df) + 1):
        for j in range(len(results_df.columns)):
            if i % 2 == 0:
                table[i, j].set_facecolor('#f0f0f0')

    intercept_val = model.params.get('const', np.nan)
    summary_text = (
        f'n = {n_obs}    '
        f'R² = {model.rsquared:.4f}    '
        f'Adj R² = {model.rsquared_adj:.4f}    '
        f'F = {model.fvalue:.2f} (p = {model.f_pvalue:.4f})    '
        f'Intercept = {intercept_val:.2f}'
    )
    if n_dropped > 0:
        summary_text += f'    [{n_dropped} obs dropped]'

    fig.text(0.05, 0.02, summary_text, fontsize=9, fontstyle='italic', color='#555555')
    fig.text(0.95, 0.02, 'Sig: *** p<0.001  ** p<0.01  * p<0.05  . p<0.10', fontsize=8, ha='right', color='#888888')

    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)


def bucket_weather_var(df, col, config):
    series = df[col].dropna()
    q = min(config['n_bins'], max(1, series.nunique()))
    if len(series) == 0 or q < 2:
        return None, [], {}

    buckets = pd.qcut(series, q=q, duplicates='drop')
    label_map = {}
    for interval in buckets.cat.categories:
        left = format(interval.left, config['fmt'])
        right = format(interval.right, config['fmt'])
        label_map[interval] = f'{left}-{right}{config["unit"]}'

    labeled = buckets.map(label_map)
    bucket_order = [label_map[iv] for iv in buckets.cat.categories]
    counts = labeled.value_counts().reindex(bucket_order).fillna(0).astype(int).to_dict()
    return labeled, bucket_order, counts


def render_heatmap(df, weather_col, weather_config, version, stadium_name, season_start, season_end, filepath, show=True):
    labeled, bucket_order, counts = bucket_weather_var(df, weather_col, weather_config)
    if labeled is None or len(bucket_order) == 0:
        return None

    stat_keys = list(BASEBALL_STATS.keys())
    stat_labels = [BASEBALL_STATS[k]['label'] for k in stat_keys]

    df_valid = df.loc[labeled.index].copy()
    df_valid['_bucket'] = labeled.values

    matrix = np.full((len(stat_keys), len(bucket_order)), np.nan)
    for i, stat in enumerate(stat_keys):
        overall_mean = df[stat].mean()
        grouped = df_valid.groupby('_bucket')[stat].mean()
        for j, bucket_label in enumerate(bucket_order):
            if bucket_label in grouped.index:
                val = grouped[bucket_label]
                matrix[i, j] = val if version == 'raw' else val - overall_mean

    if np.isnan(matrix).all():
        return None

    cmap = plt.cm.YlOrRd if version == 'raw' else plt.cm.RdBu_r
    finite_vals = matrix[np.isfinite(matrix)]
    if version == 'raw':
        norm = Normalize(vmin=np.nanmin(finite_vals), vmax=np.nanmax(finite_vals))
    else:
        bound = max(abs(np.nanmin(finite_vals)), abs(np.nanmax(finite_vals)))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0, vmax=bound)

    fig, ax = plt.subplots(figsize=(max(8, len(bucket_order) * 1.8), 5))
    im = ax.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')

    ax.set_xticks(np.arange(len(bucket_order)))
    ax.set_yticks(np.arange(len(stat_labels)))
    ax.set_xticklabels([f'{label}\n(n={counts.get(label, 0)})' for label in bucket_order], rotation=0, fontsize=9)
    ax.set_yticklabels(stat_labels, fontsize=10)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if np.isfinite(matrix[i, j]):
                ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center', fontsize=9, color='black')

    version_label = 'Raw Mean' if version == 'raw' else 'Difference from Team Mean'
    ax.set_title(f'{weather_config["label"]} Heatmap ({version_label})\n{stadium_name} ({season_start}-{season_end})', fontsize=13)
    cbar = fig.colorbar(im, ax=ax, shrink=0.9)
    cbar.set_label(weather_config['label'])
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filepath


def render_wind_line_graph(df, stat_col, stat_config, shared_bins, stadium_name, season_start, season_end, filepath, show=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    plotted = False

    for wind_col, wind_config in WIND_PROJ_VARS.items():
        series = df[[wind_col, stat_col]].dropna().copy()
        if len(series) == 0:
            continue
        series['_bin'] = pd.cut(series[wind_col], bins=shared_bins, include_lowest=True)
        grouped = series.groupby('_bin')[stat_col].agg(['mean', 'count'])
        grouped = grouped[grouped['count'] >= MIN_GAMES_PER_BIN]
        if len(grouped) == 0:
            continue

        midpoints = [(iv.left + iv.right) / 2 for iv in grouped.index]
        ax.plot(midpoints, grouped['mean'].values, marker='o', markersize=5, linewidth=2, color=wind_config['color'], label=wind_config['label'])
        plotted = True

    if not plotted:
        plt.close(fig)
        return None

    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Wind Projection (mph) — negative = blowing in, positive = blowing out', fontsize=12)
    ax.set_ylabel(stat_config['label'], fontsize=12)
    ax.set_title(f'{stat_config["label"]} by Wind Projection\n{stadium_name} ({season_start}-{season_end})', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filepath


def run_team_analysis(team_code, params_df, league_data, show=True):
    team_row = params_df[params_df['team_code'] == team_code]
    if len(team_row) == 0:
        return {'team': team_code, 'status': 'NOT FOUND', 'n_games': 0, 'n_graphics': 0}
    team_row = team_row.iloc[0]

    team_name = team_row['team_name']
    stadium_name = team_row['stadium_name']
    team_dir = os.path.join(OUTPUT_ROOT, team_code)
    os.makedirs(team_dir, exist_ok=True)

    team_data, dataset_file, season_start, season_end = prepare_team_data(team_row)
    print(f'\n{'='*60}')
    print(f'  {stadium_name} ({team_name}) — {team_code}')
    print(f'  Seasons: {season_start}-{season_end}')
    print(f'  Dataset: {dataset_file}')
    print(f'  Output: {os.path.abspath(team_dir)}')
    print(f'  Games: {len(team_data)}')
    print(f'{'='*60}')

    n_graphics = 0

    # 1. Weather distribution plots
    team_league_window = team_data[(team_data['season'] >= 2021) & (team_data['season'] <= 2025)].copy()
    weather_plots = [
        ('temp_f', 'Temperature', '°F'),
        ('wspd_mph', 'Wind Speed', ' mph'),
        ('rhum', 'Relative Humidity', '%'),
        ('pres', 'Surface Pressure', ' hPa'),
    ]

    for col, xlabel, unit in weather_plots:
        output = plot_weather_distribution(team_league_window, league_data, col, xlabel, unit, stadium_name, team_dir, show=show)
        if output is not None:
            n_graphics += 1

    # 2. OLS regressions: with/without wind, day_night as feature
    for dep_var, dep_label in DEPENDENT_VARS.items():
        for model_key, model_spec in MODEL_SPECS.items():
            try:
                model, n_dropped, _ = run_ols(team_data, dep_var, model_spec['ivars'], model_spec['center_vars'])
                results_df = results_to_dataframe(model)
                filepath = os.path.join(team_dir, f'ols_{model_key}_{dep_var}.png')
                render_regression_table(results_df, model, dep_label, model_spec['label'], int(model.nobs), n_dropped, stadium_name, season_start, season_end, filepath, show=show)
                n_graphics += 1
            except Exception as exc:
                print(f'  Regression skipped for {team_code} / {dep_var} / {model_key}: {exc}')

    # 3. Heatmaps
    for weather_col, weather_config in WEATHER_VARS.items():
        for version in ['raw', 'diff']:
            filepath = os.path.join(team_dir, f'heatmap_{weather_col}_{version}.png')
            output = render_heatmap(team_data, weather_col, weather_config, version, stadium_name, season_start, season_end, filepath, show=show)
            if output is not None:
                n_graphics += 1

    # 4. Wind projection line graphs
    all_wind_vals = pd.concat([team_data[c].dropna() for c in WIND_PROJ_VARS if c in team_data.columns], ignore_index=True)
    if len(all_wind_vals) > 0:
        shared_bins = np.linspace(all_wind_vals.min(), all_wind_vals.max(), N_WIND_BINS + 1)
        for stat_col, stat_config in BASEBALL_STATS.items():
            filepath = os.path.join(team_dir, f'wind_projection_{stat_col}.png')
            output = render_wind_line_graph(team_data, stat_col, stat_config, shared_bins, stadium_name, season_start, season_end, filepath, show=show)
            if output is not None:
                n_graphics += 1

    return {
        'team': team_code,
        'stadium': stadium_name,
        'status': 'OK',
        'dataset': dataset_file,
        'n_games': len(team_data),
        'n_graphics': n_graphics,
    }


print('Helper functions defined.')

In [ ]:
show_plots = not RUN_ALL_TEAMS
if RUN_ALL_TEAMS:
    plt.ioff()
    print(f'Batch mode: generating graphics for {len(teams_to_process)} teams')

results_summary = []
for i, team_code in enumerate(teams_to_process, start=1):
    if RUN_ALL_TEAMS:
        print(f'\n[{i}/{len(teams_to_process)}] Processing {team_code}...')
    try:
        results_summary.append(run_team_analysis(team_code, params_df, league_data, show=show_plots))
    except Exception as exc:
        print(f'  ERROR processing {team_code}: {exc}')
        results_summary.append({
            'team': team_code,
            'stadium': None,
            'status': f'ERROR: {exc}',
            'dataset': None,
            'n_games': 0,
            'n_graphics': 0,
        })

if RUN_ALL_TEAMS:
    plt.ion()

summary_df = pd.DataFrame(results_summary)
summary_path = os.path.join(OUTPUT_ROOT, 'cleaned_analysis_summary.csv')
summary_df.to_csv(summary_path, index=False)

print(f'\nSaved summary: {os.path.abspath(summary_path)}')
print(summary_df.to_string(index=False))
print(f"\nTotal graphics generated: {summary_df['n_graphics'].sum()}")